# Benchmark Analysis

Loads benchmark results and produces thesis-quality figures and tables.

**Expected outputs:**
1. Summary heatmap (Gamma by scenario x method)
2. Per-parameter recovery bar charts (membrane vs adaptation)
3. Loss convergence comparison
4. Multi-start distribution (box plots of Gamma)
5. Wall time comparison
6. Firing pattern difficulty ranking
7. LaTeX tables for thesis

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(".").resolve().parent))

from ADoptEX.benchmark import (
    load_all_results,
    summary_table,
    recovery_table,
    print_table,
    format_latex_table,
)
from ADoptEX.benchmark.scenarios import FULL_PARAMS, MEMBRANE_PARAMS

## Load Results

In [ ]:
RESULTS_DIR = Path("results")
results = load_all_results(RESULTS_DIR)
print(f"Loaded {len(results)} result files")

# Summary table
df = pd.DataFrame(summary_table(results))
df

## 1. Summary Heatmap: Gamma by Scenario x Method

In [ ]:
if not df.empty:
    pivot = df.pivot_table(index="scenario", columns="method", values="mean_gamma")
    fig, ax = plt.subplots(figsize=(12, 8))
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    # Annotate cells
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, label="Mean Gamma")
    ax.set_title("Coincidence Factor by Scenario and Method")
    plt.tight_layout()
    plt.savefig("figures/gamma_heatmap.pdf", bbox_inches="tight")
    plt.show()

## 2. Per-Parameter Recovery

In [ ]:
if not df.empty:
    rec_rows = recovery_table(results, FULL_PARAMS)
    if rec_rows:
        rec_df = pd.DataFrame(rec_rows)
        print_table(rec_rows)
    else:
        print("No recovery data (experimental scenarios only?)")

## 3. Loss Convergence Comparison

In [ ]:
# Plot loss curves for best run of each method on a representative scenario
if results:
    target_scenario = "tonic_15pct_full"
    scenario_results = [r for r in results if r.scenario_name == target_scenario]
    if scenario_results:
        fig, ax = plt.subplots(figsize=(10, 6))
        for sr in scenario_results:
            if sr.best_run and sr.best_run.loss_history:
                ax.semilogy(sr.best_run.loss_history, label=sr.method_name)
        ax.set_xlabel("Epoch / Function Evaluation")
        ax.set_ylabel("Loss")
        ax.set_title(f"Loss Convergence: {target_scenario}")
        ax.legend()
        plt.tight_layout()
        plt.savefig("figures/loss_convergence.pdf", bbox_inches="tight")
        plt.show()

## 4. Multi-Start Distribution (Box Plots)

In [ ]:
if results:
    # Collect gamma distributions per method
    method_gammas = {}
    for sr in results:
        gammas = [r.gamma for r in sr.runs if r.gamma is not None and not np.isnan(r.gamma)]
        if gammas:
            method_gammas.setdefault(sr.method_name, []).extend(gammas)
    
    if method_gammas:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.boxplot(method_gammas.values(), labels=method_gammas.keys())
        ax.set_ylabel("Gamma")
        ax.set_title("Gamma Distribution Across All Scenarios")
        ax.axhline(y=0.5, color="r", linestyle="--", alpha=0.5, label="Success threshold")
        ax.legend()
        plt.tight_layout()
        plt.savefig("figures/gamma_boxplot.pdf", bbox_inches="tight")
        plt.show()

## 5. Wall Time Comparison

In [ ]:
if not df.empty:
    time_df = df.groupby("method")["mean_wall_time_s"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(8, 5))
    time_df.plot(kind="barh", ax=ax)
    ax.set_xlabel("Mean Wall Time per Run (s)")
    ax.set_title("Computational Cost by Method")
    plt.tight_layout()
    plt.savefig("figures/wall_time.pdf", bbox_inches="tight")
    plt.show()

## 6. LaTeX Tables

In [ ]:
if not df.empty:
    # Main results table
    table_rows = summary_table(results)
    latex = format_latex_table(
        [{k: v for k, v in r.items() if k in ("scenario", "method", "mean_gamma", "best_gamma", "mean_loss", "mean_wall_time_s")}
         for r in table_rows],
        caption="Benchmark results: coincidence factor and loss across scenarios and methods"
    )
    print(latex)